In [1]:
# ==========================================================
# SINGLE-CELL: Load Moltie + pick ONE appeal + run smoke test
# - removes the in-memory 'schemas' hack
# - uses ONE PDF_PATH source of truth (no accidental override)
# - stable para_ids, no regex
# - deterministic evidence pack (first-K unless you pass para hits)
# ==========================================================

from pathlib import Path
import sys
import PyPDF2

# -----------------------
# 0) Repo roots
# -----------------------
REPO_ROOT = Path("/home/hello/Projects/Statements").resolve()
CODE_ROOT = (REPO_ROOT / "code").resolve()
assert REPO_ROOT.exists(), f"Missing repo root: {REPO_ROOT}"
assert CODE_ROOT.exists(), f"Missing code root: {CODE_ROOT}"

# ensure imports resolve to THIS repo
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

# -----------------------
# 1) Imports (real package imports, no importlib hacks)
# -----------------------
from moltie.schemas.query_object import build_atom_from_y_path
from moltie.schemas.run_config import RunConfig
from moltie.llm.verifier_prompt import build_verifier_prompt
from moltie.llm.client import verify_with_ollama, LLMClientConfig

# -----------------------
# 2) Choose appeal ONCE (only place you edit)
# -----------------------
APPEALS_DIR = (CODE_ROOT / "appeals").resolve()
assert APPEALS_DIR.exists(), f"Missing appeals dir: {APPEALS_DIR}"

APPEAL_NAME = "1._Mr_G_Lepiarz__2._Mr_D_Lewis_v__Trades_Union_Congress_-_2200228-2023___2200230-2023.pdf"  # <-- EDIT THIS ONLY

PDF_PATH = APPEALS_DIR / APPEAL_NAME
# fallback for uploaded PDFs (optional safety)
if not PDF_PATH.exists():
    alt = Path("/mnt/data") / APPEAL_NAME
    if alt.exists():
        PDF_PATH = alt

assert PDF_PATH.exists(), f"Missing PDF: {PDF_PATH}"
print("✅ Using PDF:", PDF_PATH)

# -----------------------
# 3) PDF -> text (single function)
# -----------------------
def pdf_to_text(path: Path) -> str:
    out = []
    with path.open("rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            out.append(page.extract_text() or "")
    return "\n".join(out)

doc_text = pdf_to_text(PDF_PATH)
print("PDF chars:", len(doc_text))
print("Preview:\n", doc_text[:900])

# -----------------------
# 4) Paragraph split + stable IDs (NO regex)
# -----------------------
def clean_para(s: str) -> str:
    return " ".join((s or "").strip().split())

def to_paras(text: str, min_len: int = 40):
    paras = []
    buf = []
    n = 0
    for line in (text or "").splitlines():
        if line.strip():
            buf.append(line)
        else:
            if buf:
                p = clean_para("\n".join(buf))
                buf = []
                if len(p) >= min_len:
                    n += 1
                    paras.append({"para_id": f"p{n:05d}", "text": p})
    if buf:
        p = clean_para("\n".join(buf))
        if len(p) >= min_len:
            n += 1
            paras.append({"para_id": f"p{n:05d}", "text": p})
    return paras

paras = to_paras(doc_text)
print("Paras:", len(paras))
assert paras, "No paragraphs extracted — PDF text extraction likely failed."
para_id_to_idx = {p["para_id"]: i for i, p in enumerate(paras)}

# -----------------------
# 5) Pick which atom to test
# -----------------------
Y_PATH = (REPO_ROOT / "output" / "Y_inferred.json").resolve()
assert Y_PATH.exists(), f"Missing Y file: {Y_PATH}"

ATOM_ID = "X5"  # <-- set X1..X7 here

# If you already have para hits from a prior stage, paste them here.
# MUST be para_ids from THIS doc, otherwise selection will fall back to first-K.
evidence_to_x = {}  # e.g. {"p00012": ["X5"], "p00013": ["X5"]}

atom, _ = build_atom_from_y_path(
    atom_id=ATOM_ID,
    y_path=str(Y_PATH),
    evidence_to_x=evidence_to_x,
    dedup_out_path=None,  # no write-back
)

cfg = RunConfig(
    anchors_required=2,
    thresh_score=70,
    thresh_conf=75,
    y_path=str(Y_PATH),
    y_dedup_out=None,
)

print(f"✅ Atom loaded: {atom.atom_id} x_tests={atom.x_tests}")

# -----------------------
# 6) Deterministic evidence pack (first-K OR evidence-window)
# -----------------------
K_DEBUG_PARAS = 12
NEIGHBOR_WINDOW = 3

def select_debug_paras(paras, para_id_to_idx, preferred_para_ids):
    idxs = set()
    for pid in preferred_para_ids:
        i = para_id_to_idx.get(pid)
        if i is None:
            continue
        lo = max(0, i - NEIGHBOR_WINDOW)
        hi = min(len(paras) - 1, i + NEIGHBOR_WINDOW)
        idxs.update(range(lo, hi + 1))

    if idxs:
        chosen = [paras[i] for i in sorted(idxs)]
        if len(chosen) > K_DEBUG_PARAS:
            mid = len(chosen) // 2
            half = K_DEBUG_PARAS // 2
            start = max(0, mid - half)
            chosen = chosen[start : start + K_DEBUG_PARAS]
        return chosen, {"method": "evidence_window", "note": "para_id hits + neighbors"}

    return (paras if len(paras) <= K_DEBUG_PARAS else paras[:K_DEBUG_PARAS],
            {"method": "first_k", "note": "no para_id hits; first-K fallback"})

preferred_para_ids = list(evidence_to_x.keys())
top_paras, retrieval_meta = select_debug_paras(paras, para_id_to_idx, preferred_para_ids)

evidence_pack = {
    "doc_id": PDF_PATH.stem,
    "doc_meta": {"source_path": str(PDF_PATH)},
    "paras": top_paras,
    "retrieval": {"method": retrieval_meta["method"], "note": retrieval_meta["note"], "score": None},
}

print("Evidence pack paras:", len(evidence_pack["paras"]))
print("Retrieval:", evidence_pack["retrieval"])
for p in evidence_pack["paras"][:6]:
    print(" ", p["para_id"], "|", p["text"][:150], "..." if len(p["text"]) > 150 else "")

# -----------------------
# 7) Prompt + call model
# -----------------------
prompt = build_verifier_prompt(atom=atom, evidence_pack=evidence_pack, cfg=cfg)

client_cfg = LLMClientConfig(
    model="mistral-small3.2:latest",
    ollama_url="http://localhost:11434/api/generate",
    timeout_s=180,
    temperature=0.0,
    num_predict=800,
    max_retries=2,
)

v = verify_with_ollama(prompt, client_cfg)

print("\n=== VERDICT ===")
print(v.to_dict())

print("\n=== ANCHORS ===")
for a in v.anchors:
    print(f"- {a.para_id}: {a.quote[:180]}{'...' if len(a.quote)>180 else ''}")
    print(f"  why: {a.why_it_matters}")

✅ Using PDF: /home/hello/Projects/Statements/code/appeals/1._Mr_G_Lepiarz__2._Mr_D_Lewis_v__Trades_Union_Congress_-_2200228-2023___2200230-2023.pdf
PDF chars: 152818
Preview:
 Case Number s: 2200228/2023  and 22 00230/2023  
1 
 
 
 
EMPLOYMENT TRIBUNALS  
 
 
Claimants:   (1) Mr G Lepiarz  
  (2) Mr D Lewis  
 
Respondent:   Trade s Union Congress  
 
 
Heard at:    London Central (via Cloud Video Platform)        
   On: 27, 28 and 29 February 2024 and  1, 4, 5, 6 and 7 March 2024   
 
Before:   Employment Judge Joffe  
    Mr M Simon  
    Mr F Benson     
 
Representation  
Claimant s:  Mr P Livingston, counsel   
Respondent:   Ms K Annand, counsel  
 
 
RESERVED  JUDGMENT  
 
 
1. The claimants’ claims of unfair dismissal are well-founded  and are upheld.  
2. The claimants caused or contributed to their dismissals by blameworthy 
conduct and it is just and equitable to reduce the compensatory award payable 
to the claimant s by: 
a. First claimant: 15%;  
b. Second claimant: 20%.

### Moltie agent execution (single document)

This cell is the **minimal execution entry point** for Moltie.

It invokes the Moltie agent loop on:
- one document (identified by `doc_id`)
- pre-split paragraphs with stable `para_id`s
- a fully constructed `AtomQuery`
- runtime and LLM configuration

The agent loop evaluates the evidence and returns exactly one outcome:
- a **Verdict** if the atom is supported, or
- a **NegativeExit** if the agent cannot validate it.

This cell assumes all inputs are valid and integrated.
It represents the **canonical way Moltie is run**.


# Moltie — Single‑Document Flow

This file explains **what runs and in what order** when Moltie evaluates **one document against one atom**.

---

## Entry point

You call **one function only**:

```python
res = run_agent_on_one_doc(doc_id, paras, atom, run_cfg, llm_cfg)
```

The result is **exactly one** of:
- `Verdict` → accepted evidence‑anchored result  
- `NegativeExit` → explicit, audited stop condition

---

## What happens inside

1) **Retrieve (recall only)**  
`retrieve_windowed_evidence`  
Selects a small window of paragraphs and tracks coverage.

2) **Ask (build prompt)**  
`build_verifier_prompt`  
Compiles AtomQuery + evidence + strict Verdict JSON contract.

3) **Verify (LLM single‑shot)**  
`verify_with_ollama`  
Calls the LLM once and returns a schema‑valid `Verdict` (repair/retry if needed).

4) **Truth gate**  
Agent checks every anchor:
- `para_id` exists  
- quote is verbatim  
Failure ⇒ reject.

5) **Score progress**  
Agent computes a quality metric to detect improvement or plateau.

6) **Decide**  
- Accept ⇒ return `Verdict`  
- Plateau / exhausted ⇒ return `NegativeExit`  
- Else ⇒ refine query and loop

---

## Key rule

**The agent controls the loop.  
The LLM answers one question.**


In [2]:
# ==========================================================
# MOLTIE SINGLE-DOC RUN HARNESS (REAL LOOP, NO FAKE evidence_to_x)
#  - pins client_cfg deterministically (num_predict + stop + timeout)
#  - prints config so truncation is impossible to miss
#  - FORENSIC: prints invalid_anchors details + quote vs paragraph text
# ==========================================================

import sys, importlib, json
from pathlib import Path
from pprint import pprint

# -------------------------
# 0) Explicit repo anchors
# -------------------------
REPO_ROOT = Path("/home/hello/Projects/Statements").resolve()
CODE_ROOT = (REPO_ROOT / "code").resolve()
assert REPO_ROOT.exists(), f"Missing repo root: {REPO_ROOT}"
assert CODE_ROOT.exists(), f"Missing code root: {CODE_ROOT}"

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

# -------------------------
# 1) Import modules (then reload in dependency order)
# -------------------------
import moltie.schemas.run_config as rc_mod
import moltie.llm.verifier_prompt as vp_mod
import moltie.llm.client as client_mod
import moltie.schemas.query_object as qo_mod
import moltie.agent.loop as loop_mod

importlib.reload(rc_mod)      # config first
importlib.reload(vp_mod)      # prompt
importlib.reload(client_mod)  # client boundary
importlib.reload(qo_mod)      # AtomQuery helpers
importlib.reload(loop_mod)    # loop last

RunConfig = rc_mod.RunConfig
AtomQuery = qo_mod.AtomQuery
run_agent_on_one_doc = loop_mod.run_agent_on_one_doc
LLMClientConfig = client_mod.LLMClientConfig  # <-- pin type from reloaded module

print("[moltie.debug] repo:", REPO_ROOT)
print("[moltie.debug] code:", CODE_ROOT)
print("[moltie.debug] loop file:", Path(loop_mod.__file__).resolve())

# -------------------------
# 2) REQUIRED INPUTS (must already exist in notebook)
# -------------------------
for name in ("PDF_PATH", "paras"):
    if name not in globals():
        raise RuntimeError(f"Missing required notebook variable: {name}")

assert isinstance(PDF_PATH, Path), "PDF_PATH must be a pathlib.Path"
assert PDF_PATH.exists(), f"Missing PDF file: {PDF_PATH}"
assert isinstance(paras, list) and len(paras) > 0, "paras must be non-empty list"
assert isinstance(paras[0], dict) and "para_id" in paras[0] and "text" in paras[0], "paras must be list of dicts with para_id/text"

# --- ensure notebook paras match loop rechunking ---
paras = loop_mod._maybe_rechunk_single_blob_paras(paras)

DOC_ID = PDF_PATH.stem

print("[moltie.debug] doc_id:", DOC_ID)
print("[moltie.debug] paras:", len(paras), "sample:", paras[0]["para_id"], "len:", len(paras[0]["text"] or ""))

# -------------------------
# 2b) PIN LLM CLIENT CONFIG (prevents truncation + notebook state leaks)
# -------------------------
_client_kwargs = dict(
    model="mistral-small3.2:latest",
    ollama_url="http://localhost:11434/api/generate",
    timeout_s=180,
    temperature=0.0,   # fallback if iter_temp_enabled=False
    num_predict=1000,
    max_retries=2,
)

# optional stop pin (only if supported by the dataclass)
try:
    if "stop" in getattr(LLMClientConfig, "__annotations__", {}):
        _client_kwargs["stop"] = []
except Exception:
    pass

client_cfg = LLMClientConfig(**_client_kwargs)

print("[moltie.debug] client_cfg.model:", getattr(client_cfg, "model", None))
print("[moltie.debug] client_cfg.num_predict:", getattr(client_cfg, "num_predict", None))
print("[moltie.debug] client_cfg.timeout_s:", getattr(client_cfg, "timeout_s", None))
print("[moltie.debug] client_cfg.temperature(fallback):", getattr(client_cfg, "temperature", None))
if hasattr(client_cfg, "stop"):
    print("[moltie.debug] client_cfg.stop:", getattr(client_cfg, "stop", None))

# -------------------------
# 3) Load Y JSON (real file)
# -------------------------
Y_PATH = REPO_ROOT / "output" / "Y_inferred.json"
assert Y_PATH.exists(), f"Missing Y file: {Y_PATH}"
y_json = json.loads(Y_PATH.read_text(encoding="utf-8"))

x_tests_map = (y_json.get("x_tests") or {})
assert isinstance(x_tests_map, dict) and len(x_tests_map) > 0, "Y JSON has no x_tests"

print("[moltie.debug] Y file:", Y_PATH)
print("[moltie.debug] Y x_tests:", len(x_tests_map))

# -------------------------
# 4) Run loop for each X test in Y (real AtomQuery construction)
# -------------------------
results = []

# --- OPTIONAL: temperature schedule toggle (runner-level) ---
# Set these to your desired behavior.
ITER_TEMP_ENABLED = True
ITER_TEMP_START = 0.20   # iter=1 (more exploratory)
ITER_TEMP_END = 0.00     # iter=max_iters (more deterministic)
ITER_TEMP_CURVE = "linear"  # "linear" or "exp"
ITER_TEMP_CAP = 0.80

print("[moltie.debug] iter_temp_enabled:", ITER_TEMP_ENABLED)
if ITER_TEMP_ENABLED:
    print("[moltie.debug] iter_temp_start/end/curve/cap:", ITER_TEMP_START, ITER_TEMP_END, ITER_TEMP_CURVE, ITER_TEMP_CAP)

for x_key in sorted(x_tests_map.keys()):
    print("\n==================================================")
    print("Running atom:", x_key, "->", x_tests_map[x_key].get("name"))
    print("==================================================")

    merged = qo_mod.merge_indicators_and_excludes(y_json, [x_key])

    atom = AtomQuery(
        atom_id=x_key,
        x_tests=[x_key],
        proposition=x_tests_map[x_key].get("name", x_key),
        positive_indicators=merged["positive_indicators"],
        excludes=merged["excludes"],
        keyword_seeds=merged["positive_indicators"],
        expansion_terms=[],
    )

    cfg2 = RunConfig.from_dict({
        "debug": False,
        "harvest_mode": False,

        "max_iters": 3,
        "window_size": 12,
        "stride": 6,
        "top_windows": 3,
        "k_chunks_per_doc": 12,

        "anchors_required": 1,
        "min_hits": 1,
        "thresh_score": 1,
        "thresh_conf": 0.8,   # = 80% because loop normalizes 0..1 -> /100

        "plateau_p": 2,
        "eps_improve": 0,

        # NEW: per-iteration temperature schedule (only used by new loop.py)
        "iter_temp_enabled": ITER_TEMP_ENABLED,
        "iter_temp_start": ITER_TEMP_START,
        "iter_temp_end": ITER_TEMP_END,
        "iter_temp_curve": ITER_TEMP_CURVE,
        "iter_temp_cap": ITER_TEMP_CAP,
    })

    res = run_agent_on_one_doc(
        DOC_ID,
        paras,
        atom,
        cfg2,
        client_cfg,
    )

    results.append((x_key, res))

    print("\nRESULT for", x_key)
    if res.verdict:
        v = res.verdict.to_dict()
        print("relevant=", v.get("relevant"),
              "score=", v.get("precedent_score"),
              "conf=", v.get("confidence"),
              "anchors=", len(v.get("anchors") or []))
        pprint(v.get("anchors"))
    else:
        print("NEGATIVE:", res.negative_exit.reason if res.negative_exit else None)

        # --- FORENSIC: find FIRST invalid_anchors and print quote vs paragraph text ---
        found = False
        for t in (res.trace or []):
            err = (t.get("error") or {})
            if err.get("type") != "invalid_anchors":
                continue

            found = True
            print("\n[FORENSIC] iter:", t.get("iter"))
            print("[FORENSIC] bad_details:", err.get("details"))

            v = t.get("verdict") or {}
            anchors = v.get("anchors") or []
            print("[FORENSIC] anchors_from_model_count:", len(anchors))
            pprint(anchors[:3])

            bad_details = err.get("details") or []
            for typ, pid in bad_details[:3]:
                print(f"\n[FORENSIC] bad_type={typ} pid={pid}")

                a_hits = [a for a in anchors if isinstance(a, dict) and a.get("para_id") == pid]
                if not a_hits:
                    print("[FORENSIC] model_anchor_for_pid: NONE (pid mismatch or missing)")
                else:
                    q0 = (a_hits[0].get("quote") or "")
                    print("[FORENSIC] model_quote_repr:", repr(q0)[:1200])

                ptxt = next((p.get("text", "") for p in paras if p.get("para_id") == pid), "")
                print("[FORENSIC] global_para_text_len:", len(ptxt))
                print("[FORENSIC] global_para_text_repr:", repr(ptxt)[:1600])

                if a_hits:
                    q0 = (a_hits[0].get("quote") or "")
                    print("[FORENSIC] raw_contains:", (q0 in ptxt))

            break

        if not found:
            print("[FORENSIC] no invalid_anchors entries in trace (different failure mode).")

# -------------------------
# 5) Output (compact + useful)
# -------------------------
print("\n================ RESULT ================")
if results and results[-1][1].verdict:
    v = results[-1][1].verdict.to_dict()
    print("VERDICT: relevant=", v.get("relevant"),
          "score=", v.get("precedent_score"),
          "conf=", v.get("confidence"),
          "anchors=", len(v.get("anchors") or []),
          "matched_X=", v.get("matched_X"))
    print("\nanchors:")
    pprint(v.get("anchors"))
else:
    print("NEGATIVE_EXIT:")
    last_res = results[-1][1] if results else None
    pprint(last_res.negative_exit.to_dict() if (last_res and last_res.negative_exit) else None)

print("\n================ TRACE (last 3 entries) ================")
last_res = results[-1][1] if results else None
pprint((last_res.trace or [])[-3:] if last_res else None)

print("\n================ TRACE SUMMARY ================")
print("iters:", (last_res.iters if last_res else None), "trace_len:", (len(last_res.trace) if last_res else None))

[moltie.debug] repo: /home/hello/Projects/Statements
[moltie.debug] code: /home/hello/Projects/Statements/code
[moltie.debug] loop file: /home/hello/Projects/Statements/code/moltie/agent/loop.py
[moltie.debug] doc_id: 1._Mr_G_Lepiarz__2._Mr_D_Lewis_v__Trades_Union_Congress_-_2200228-2023___2200230-2023
[moltie.debug] paras: 377 sample: p00001 len: 47
[moltie.debug] client_cfg.model: mistral-small3.2:latest
[moltie.debug] client_cfg.num_predict: 1000
[moltie.debug] client_cfg.timeout_s: 180
[moltie.debug] client_cfg.temperature(fallback): 0.0
[moltie.debug] Y file: /home/hello/Projects/Statements/output/Y_inferred.json
[moltie.debug] Y x_tests: 6
[moltie.debug] iter_temp_enabled: True
[moltie.debug] iter_temp_start/end/curve/cap: 0.2 0.0 linear 0.8

Running atom: X1 -> Unauthorized Work Claim
[moltie.loop] start doc_id='1._Mr_G_Lepiarz__2._Mr_D_Lewis_v__Trades_Union_Congress_-_2200228-2023___2200230-2023' atom_id='X1' n_paras=377

[moltie.client] ===== Attempt A =====
[moltie.client] at

RuntimeError: verify_with_ollama failed after 3 attempts. Last error: Could not locate JSON object braces in model output. Last output (truncated): '{"atom_id": "Z2V0X2F0b21fc29tZV9hZ2VudGl0eV9kZXNpZGVudGlhbH1fYXRvbV9hZ2VudGl0eV9kZXNpZGVudGlhbH1fYXRvbV9hZ2VudGl0eV9kZXNpZGVudGlhbH1fYXRvbV9hZ2VudGl0eV9kZXNpZGVudGlhbH1fYXRvbV9hZ2VudGl0eV9kZXNpZGVudGlhbH1fYXRvbV9hZ2VudGl0eV9kZXNpZGVudGlhbH1fYXRvbV9hZ2VudGl0eV9kZXNpZGVudGlhbH1fYXRvbV9hZ2VudGl0eV9kZXNpZGVudGlhbH1fYXRvbV9hZ2VudGl0eV9kZXNpZGVudGlhbH1fYXRvbV9hZ2VudGl0eV9kZXNpZGVudGlhbH1fYXRvbV9hZ2VudGl0eV9kZXNpZGVudGlhbH1fYXRvbV9hZ2VudGl0eV9kZXNpZGVudGlhbH1fYXRvbV9hZ2VudGl0eV9kZXNpZGVudGlhbH1fYXRvbV9hZ2VudGl0eV9kZXNpZGVudGlhbH1fYXRvbV9hZ2VudGl0eV9kZXNpZGVudGlhbH1fYXRvbV9hZ2VudGl0eV9kZXNpZGVudGlhbH1fYXRvbV9hZ2VudGl0eV9kZXNpZGVudGlhbH1fYXRvbV9hZ2VudGl0eV9kZXNpZGVudGlhbH1fYXRvbV9hZ2VudGl0eV9kZXNpZGVudGlhbH1fYXRvbV9hZ2VudGl0eV9kZXNpZGVudGlhbH1fYXRvbV9hZ2VudGl0eV9kZXNpZGVudGlhbH1fYXRvbV9hZ2VudGl0eV9kZXNpZGVudGlhbH1fYXRvbV9hZ2VudGl0eV9kZXNpZGVudGlhbH

In [ ]:
print("\n================ END OF RUN ================")